In [0]:
WITH bin_mapping AS (
  SELECT * FROM VALUES
    ('TAG $5 - 20 CR', '$5'),
    ('BP TAG+$10 - 52 CR', '$10/$15'), ('PREM TAG+$10 - 54 CR', '$10/$15'), ('TAG $10 - 40 CR', '$10/$15'),
    ('BP TAG+$15 - 72 CR', '$10/$15'), ('PREM TAG+$15 - 74 CR', '$10/$15'), ('TAG $15 - 60 CR', '$10/$15'),
    ('BP TAG+$25 - 172 CR', '$25'), ('PREM TAG+$25 - 174 CR', '$25'), ('TAG $25 - 160 CR', '$25'),
    ('BP TAG+$35 - 257 CR', '$35'), ('PREM TAG+$35 - 257 CR', '$35'), ('PREM TAG+$35 - 259 CR', '$35'), ('TAG $35 - 245 CR', '$35'),
    ('BP TAG+$50 - 377 CR', '$50'), ('PREM TAG+$50 - 379 CR', '$50'), ('TAG $50 - 365 CR', '$50'),
    ('BP TAG+$75 - 577 CR', '$75'), ('PREM TAG+$75 - 579 CR', '$75'), ('TAG $75 - 565 CR', '$75'),
    ('BP TAG+$100 - 802 CR', '$100/$129'), ('PREM TAG+$100 - 804 CR', '$100/$129'), ('PROMO BP TAG $100 - 1002 CR', '$100/$129'), ('TAG $100 - 790 CR', '$100/$129'),
    ('BP TAG+$129 - 1107 CR', '$100/$129'), ('PREM TAG+$129 - 1109 CR', '$100/$129'), ('TAG $129 - 1095 CR', '$100/$129')
  AS t(Item_Description, price_bin)
),

conditions AS (
  SELECT * FROM VALUES
    (2026, 'SP Offer Period', '2026-05-18', '2026-09-07'),
    (2026, 'SP Pre Period',   '2026-04-13', '2026-05-17'),
    (2025, 'SP Offer Period', '2025-05-18', '2025-09-07'),
    (2025, 'SP Pre Period',   '2025-04-13', '2025-05-17')
  AS c(Year, Period, date_start, date_end)
),

date_filtered AS (
  SELECT dateid, datefull
  FROM cpx_dataanalytics_gold.customer360.cpx_d_date
  WHERE datefull BETWEEN '2025-04-13' AND '2026-09-07'
),

location_filtered AS (
  SELECT LocationID, LC_Trade_Brand_Description, LC_Location_Type_Description
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description LIKE '%Restaurant%'
),

game_band AS (
  SELECT
    d.datefull,
    l.LC_Trade_Brand_Description,
    bm.price_bin,
    t.Net_Revenue,
    t.Quantity,
    t.SCENEFLAG
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  INNER JOIN date_filtered d ON d.dateid = t.VisitDateId
  INNER JOIN location_filtered l ON l.LocationID = t.LocationId
  INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON i.itemskey = t.ItemSkey
  INNER JOIN bin_mapping bm ON bm.Item_Description = i.Item_Description
),

tagged AS (
  SELECT
    CASE WHEN c.Period LIKE '%Pre%' THEN 'Pre' ELSE 'Offer' END AS Period_Type,
    c.Year,
    g.LC_Trade_Brand_Description,
    g.price_bin,
    g.Net_Revenue,
    g.Quantity,
    g.SCENEFLAG
  FROM game_band g
  INNER JOIN conditions c
    ON c.Year = YEAR(g.datefull)
    AND g.datefull BETWEEN c.date_start AND c.date_end
),

base_data AS (
  SELECT
    Period_Type,
    Year,
    LC_Trade_Brand_Description,
    price_bin,
    SUM(Net_Revenue) AS Total_Revenue,
    SUM(Quantity) AS Total_Sales,
    SUM(CASE WHEN SCENEFLAG = 'Yes' THEN Net_Revenue ELSE 0 END) AS Scene_Revenue,
    SUM(CASE WHEN SCENEFLAG = 'Yes' THEN Quantity ELSE 0 END) AS Scene_Sales
  FROM tagged
  GROUP BY Period_Type, Year, LC_Trade_Brand_Description, price_bin
),

totals AS (
  SELECT
    Period_Type, Year, LC_Trade_Brand_Description,
    SUM(Total_Revenue) AS Grand_Total_Revenue
  FROM base_data
  GROUP BY Period_Type, Year, LC_Trade_Brand_Description
),

shares AS (
  SELECT
    b.Period_Type, b.Year, b.LC_Trade_Brand_Description, b.price_bin,
    b.Total_Revenue / t.Grand_Total_Revenue AS Share
  FROM base_data b
  JOIN totals t
    ON b.Period_Type = t.Period_Type AND b.Year = t.Year
    AND b.LC_Trade_Brand_Description = t.LC_Trade_Brand_Description
),

shift AS (
  SELECT
    s_offer.Year,
    s_offer.LC_Trade_Brand_Description,
    s_offer.price_bin,
    s_pre.Share AS Pre_Share,
    s_offer.Share AS Offer_Share,
    s_offer.Share - s_pre.Share AS Shift
  FROM shares s_offer
  JOIN shares s_pre
    ON s_offer.Year = s_pre.Year
    AND s_offer.LC_Trade_Brand_Description = s_pre.LC_Trade_Brand_Description
    AND s_offer.price_bin = s_pre.price_bin
    AND s_pre.Period_Type = 'Pre'
  WHERE s_offer.Period_Type = 'Offer'
)

SELECT
  y2025.LC_Trade_Brand_Description,
  y2025.price_bin,
  y2025.Pre_Share AS Pre_Share_2025,
  y2025.Offer_Share AS Offer_Share_2025,
  y2025.Shift AS Normal_Shift_2025,
  y2026.Pre_Share AS Pre_Share_2026,
  y2026.Offer_Share AS Offer_Share_2026,
  y2026.Shift AS Actual_Shift_2026,
  y2026.Shift - y2025.Shift AS Excess_Shift
FROM shift y2025
JOIN shift y2026
  ON y2025.LC_Trade_Brand_Description = y2026.LC_Trade_Brand_Description
  AND y2025.price_bin = y2026.price_bin
  AND y2025.Year = 2025 AND y2026.Year = 2026
ORDER BY LC_Trade_Brand_Description, price_bin;